# SoSe26 Case Study - Group 38

**Participants:** *(add names)*

## Table of Contents

1. [Importing the data](#1-importing-the-data)
2. [Data preparation](#2-data-preparation)
3. [Creation of the final dataset](#3-creation-of-the-final-dataset)
4. [Evaluation and Result](#4-evaluation-and-result)

---

## Task

We recommend the **most popular vehicle** by choosing the most popular
**component series** in each category **K1-K7**, measured by **KBA
registrations**. A series is designation + manufacturer + plant
(for example `K1BE1-104-1041`). We also look at each year for trends.

**How we read an ID.** `K1BE1-101-1011-7` means:

- designation / type: `K1BE1`
- manufacturer: `101`
- plant: `1011`
- sequential number: `7` (one physical unit - not ranked)

We rank the **series** `K1BE1-101-1011`, not the serial number.


## 1. Importing the data

### Interpretation

Management wants a future vehicle built from components customers already
chose. Popularity = how many **registered** vehicles contain that series
(type + manufacturer + plant).

A finished car has four installed components: engine (K1), seats (K2), gearbox
(K3), body (K4 **or** K5 **or** K6 **or** K7). K4-K7 cannot be combined.

### Data-selection strategy

| Use | Folder / files | Why |
|-----|----------------|-----|
| Yes | `Fahrzeug/Bestandteile_Fahrzeuge_*` | installed component IDs with manufacturer and plant |
| Yes | `Zulassungen/Zulassungen_alle_Fahrzeuge.csv` | registration year for counts and trends |
| Yes | `Fahrzeug/Fahrzeuge_*`, `Komponente/*`, `Einzelteil/*` | defect flags for the second analysis mode |
| No | `Logistikverzug` | General Task 1 only |

Join key: `Bestandteile.ID_Fahrzeug` = `Zulassungen.IDNummer`.

**Defectiveness rule:** a vehicle counts as defective if **any** of these is
marked defective:

1. the vehicle itself,
2. an installed component (engine, seats, gearbox, or body), or
3. a single part built into that component.

If a part is defective, the component that contains it is also treated as
defective. Only defects with a date **after** the vehicle’s registration are
used. The analysis (and the app) can switch between all registrations and
counts that exclude these defective vehicles.


In [28]:
# setup: suppress warnings, load libraries, define paths and labels

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display

# relative paths under Data/
DATA_DIR = Path("Data")
FINAL_CSV = DATA_DIR / "SoSe26_Case_Study_finalData_Group_38.csv"

# four vehicle parts lists: (relative path, brand, model)
BOM_FILES = [
    ("Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ11.csv", "OEM1", "Typ11"),
    ("Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ12.csv", "OEM1", "Typ12"),
    ("Fahrzeug/Bestandteile_Fahrzeuge_OEM2_Typ21.csv", "OEM2", "Typ21"),
    ("Fahrzeug/Bestandteile_Fahrzeuge_OEM2_Typ22.csv", "OEM2", "Typ22"),
]

# parts-list columns mapped to K categories (body category comes from the ID prefix)
SLOTS = [("ID_Motor", "K1"), ("ID_Sitze", "K2"), ("ID_Schaltung", "K3"), ("ID_Karosserie", "body")]

# plain-language labels for the charts (by component type)
TYPE_LABEL = {
    "K1BE1": "Petrol engine (OEM1)",
    "K1BE2": "Petrol engine (OEM2)",
    "K1DI1": "Diesel engine (OEM1)",
    "K1DI2": "Diesel engine (OEM2)",
    "K2ST1": "Fabric seats (OEM1)",
    "K2ST2": "Fabric seats (OEM2)",
    "K2LE1": "Leather seats (OEM1)",
    "K2LE2": "Leather seats (OEM2)",
    "K3SG1": "Manual gearbox (OEM1)",
    "K3SG2": "Manual gearbox (OEM2)",
    "K3AG1": "Automatic gearbox (OEM1)",
    "K3AG2": "Automatic gearbox (OEM2)",
    "K4": "Body (OEM1 Typ11)",
    "K5": "Body (OEM1 Typ12)",
    "K6": "Body (OEM2 Typ21)",
    "K7": "Body (OEM2 Typ22)",
}

PLOTLY_BLUES = ["#2F5F7A", "#5BA4CF", "#8FCBE8", "#C5E4F3"]
CATEGORIES = ["K1", "K2", "K3", "K4", "K5", "K6", "K7"]


def read_semicolon_csv(path, **kwargs):
    # original tables use semicolon separators
    return pd.read_csv(path, sep=";", **kwargs)


def parse_component_id(series: pd.Series) -> pd.DataFrame:
    # split ID into type, manufacturer, plant (serial is ignored)
    parts = series.astype(str).str.split("-", n=3, expand=True)
    out = pd.DataFrame({
        "component_type": parts[0],
        "manufacturer": parts[1],
        "plant": parts[2],
    })
    out["series"] = out["component_type"] + "-" + out["manufacturer"] + "-" + out["plant"]
    return out


In [29]:
# check that required input files exist under Data/

for rel, oem, vtype in BOM_FILES:
    path = DATA_DIR / rel
    print(f"{'OK' if path.exists() else 'MISSING':7}  {oem} {vtype:5}  {path}")

zul_path = DATA_DIR / "Zulassungen" / "Zulassungen_alle_Fahrzeuge.csv"
print(f"{'OK' if zul_path.exists() else 'MISSING':7}  registrations  {zul_path}")


OK       OEM1 Typ11  Data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ11.csv
OK       OEM1 Typ12  Data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ12.csv
OK       OEM2 Typ21  Data/Fahrzeug/Bestandteile_Fahrzeuge_OEM2_Typ21.csv
OK       OEM2 Typ22  Data/Fahrzeug/Bestandteile_Fahrzeuge_OEM2_Typ22.csv
OK       registrations  Data/Zulassungen/Zulassungen_alle_Fahrzeuge.csv


### Load vehicle parts lists and inspect structure

Each row is one vehicle. Four ID columns are the installed engine, seats,
gearbox and body. Each ID already encodes manufacturer and plant.


In [30]:
# load only the ID columns needed for the analysis
bom_frames = []
for rel, oem, vtype in BOM_FILES:
    path = DATA_DIR / rel
    df = read_semicolon_csv(
        path,
        usecols=["ID_Fahrzeug", "ID_Karosserie", "ID_Schaltung", "ID_Sitze", "ID_Motor"],
        dtype=str,
    )
    df["oem"] = oem
    df["vehicle_type"] = vtype
    # quality check: row count, duplicate keys, missing cells
    print(f"{oem} {vtype}: {len(df):,} rows, duplicate vehicle IDs={df['ID_Fahrzeug'].duplicated().sum()}, missing cells={int(df.isna().sum().sum())}")
    bom_frames.append(df)

# combine all OEMs / vehicle types into one table
bom = pd.concat(bom_frames, ignore_index=True)
print(f"\nTotal vehicles in parts lists: {len(bom):,}  unique IDs: {bom['ID_Fahrzeug'].nunique():,}")
bom.head()


OEM1 Typ11: 1,977,164 rows, duplicate vehicle IDs=0, missing cells=0
OEM1 Typ12: 408,096 rows, duplicate vehicle IDs=0, missing cells=0
OEM2 Typ21: 512,354 rows, duplicate vehicle IDs=0, missing cells=0
OEM2 Typ22: 306,490 rows, duplicate vehicle IDs=0, missing cells=0

Total vehicles in parts lists: 3,204,104  unique IDs: 3,204,104


,ID_Karosserie,ID_Schaltung,ID_Sitze,ID_Motor,ID_Fahrzeug,oem,vehicle_type
0,K4-112-1121-3,K3SG1-105-1051-32,K2LE1-109-1091-2,K1BE1-101-1011-7,11-1-11-1,OEM1,Typ11
1,K4-112-1121-4,K3SG1-105-1051-141,K2ST1-109-1092-5,K1BE1-101-1011-12,11-1-11-2,OEM1,Typ11
2,K4-112-1121-7,K3SG1-105-1051-106,K2ST1-109-1092-57,K1BE1-101-1011-38,11-1-11-3,OEM1,Typ11
3,K4-112-1121-9,K3SG1-105-1051-21,K2ST1-109-1092-91,K1BE1-101-1011-97,11-1-11-4,OEM1,Typ11
4,K4-112-1121-11,K3SG1-105-1051-59,K2ST1-109-1092-4,K1BE1-101-1011-65,11-1-11-5,OEM1,Typ11


### ID pattern

designation - manufacturer - plant - sequential number.
Example: `K1BE1-101-1011-7` -> series `K1BE1-101-1011` (type + manufacturer + plant).


In [31]:
# show how a full ID maps to type, manufacturer, plant and series
sample = bom[["ID_Fahrzeug", "ID_Motor"]].head(5).copy()
parsed = parse_component_id(sample["ID_Motor"])
sample = pd.concat([sample, parsed], axis=1)
sample


,ID_Fahrzeug,ID_Motor,component_type,manufacturer,plant,series
0,11-1-11-1,K1BE1-101-1011-7,K1BE1,101,1011,K1BE1-101-1011
1,11-1-11-2,K1BE1-101-1011-12,K1BE1,101,1011,K1BE1-101-1011
2,11-1-11-3,K1BE1-101-1011-38,K1BE1,101,1011,K1BE1-101-1011
3,11-1-11-4,K1BE1-101-1011-97,K1BE1,101,1011,K1BE1-101-1011
4,11-1-11-5,K1BE1-101-1011-65,K1BE1,101,1011,K1BE1-101-1011


### Load registrations

KBA table: vehicle ID, municipality, registration date. Popularity uses the
date (year). Municipality is not required for the series ranking.


In [32]:
# load KBA registrations: vehicle ID, municipality, registration date
zul = read_semicolon_csv(zul_path, usecols=["IDNummer", "Gemeinden", "Zulassung"])
print(zul.dtypes)
print(f"rows={len(zul):,}  duplicate IDs={zul['IDNummer'].duplicated().sum()}  missing={zul.isna().sum().to_dict()}")

# extract registration year for the trend analysis
zul["Zulassung"] = pd.to_datetime(zul["Zulassung"], errors="coerce")
zul["year"] = zul["Zulassung"].dt.year.astype("Int64")
print("registration years:", sorted(zul["year"].dropna().unique().tolist()))
print("dates with parse problems:", int(zul["year"].isna().sum()))

# plot registrations per year
year_counts = zul["year"].value_counts().sort_index()
fig = px.bar(
    x=year_counts.index.astype(int),
    y=year_counts.values,
    title="Registered vehicles per year",
    labels={"x": "Registration year", "y": "Vehicles"},
    color_discrete_sequence=["#5BA4CF"],
)
fig.update_layout(font_family="Source Sans Pro", plot_bgcolor="white", paper_bgcolor="white")
fig.show()
zul.head()


IDNummer     object
Gemeinden    object
Zulassung    object
dtype: object
rows=3,204,104  duplicate IDs=0  missing={'IDNummer': 0, 'Gemeinden': 0, 'Zulassung': 0}
registration years: [2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016]
dates with parse problems: 0


,IDNummer,Gemeinden,Zulassung,year
0,11-1-11-1,DRESDEN,2009-01-01,2009
1,11-1-11-2,DRESDEN,2009-01-01,2009
2,12-1-12-1,LEIPZIG,2009-01-01,2009
3,12-1-12-2,LEIPZIG,2009-01-01,2009
4,12-1-12-3,DORTMUND,2009-01-01,2009


## 2. Data preparation

Checks before transforming: types, missing values, duplicates, join success.
Then parse type, manufacturer and plant from each ID and reshape to tidy
(long) form: one row per vehicle x category.


In [33]:
# check dtypes, then join parts lists to registrations on vehicle ID
print("Parts-list dtypes:\n", bom.dtypes)
print("\nRegistration dtypes after parse:\n", zul.dtypes)

# indicator=True reports whether each vehicle found a registration
merged_check = bom.merge(
    zul[["IDNummer", "year"]],
    left_on="ID_Fahrzeug",
    right_on="IDNummer",
    how="left",
    indicator=True,
)
print("\nMerge result (left = parts list, right = registrations):")
print(merged_check["_merge"].value_counts())
match_rate = (merged_check["_merge"] == "both").mean()
print(f"Match rate: {match_rate:.1%}")
del merged_check


Parts-list dtypes:
 ID_Karosserie    object
ID_Schaltung     object
ID_Sitze         object
ID_Motor         object
ID_Fahrzeug      object
oem              object
vehicle_type     object
dtype: object

Registration dtypes after parse:
 IDNummer             object
Gemeinden            object
Zulassung    datetime64[ns]
year                  Int64
dtype: object

Merge result (left = parts list, right = registrations):
_merge
both          3204104
left_only           0
right_only          0
Name: count, dtype: int64
Match rate: 100.0%


In [34]:
# tidy long form + aggregate per file (keeps RAM lower on large tables)
# one row in the result = year x oem x vehicle_type x category x series
agg_parts = []
zul_slim = zul[["IDNummer", "year"]].dropna().copy()
zul_slim["year"] = zul_slim["year"].astype(int)

for rel, oem, vtype in BOM_FILES:
    bom_one = read_semicolon_csv(
        DATA_DIR / rel,
        usecols=["ID_Fahrzeug", "ID_Karosserie", "ID_Schaltung", "ID_Sitze", "ID_Motor"],
        dtype=str,
    )
    m = bom_one.merge(zul_slim, left_on="ID_Fahrzeug", right_on="IDNummer", how="inner")
    for col, default_cat in SLOTS:
        parsed = parse_component_id(m[col])
        out = pd.DataFrame({
            "year": m["year"].values,
            "oem": oem,
            "vehicle_type": vtype,
            "component_type": parsed["component_type"].values,
            "manufacturer": parsed["manufacturer"].values,
            "plant": parsed["plant"].values,
            "series": parsed["series"].values,
        })
        out["category"] = out["component_type"] if default_cat == "body" else default_cat
        g = (
            out.groupby(
                ["year", "oem", "vehicle_type", "category", "component_type", "manufacturer", "plant", "series"],
                as_index=False,
            )
            .size()
            .rename(columns={"size": "n_registrations"})
        )
        agg_parts.append(g)
    print(f"{oem} {vtype}: matched {len(m):,} registered vehicles")

final = pd.concat(agg_parts, ignore_index=True)
final = final.groupby(
    ["year", "oem", "vehicle_type", "category", "component_type", "manufacturer", "plant", "series"],
    as_index=False,
)["n_registrations"].sum()
print(f"Aggregated rows before labels: {len(final):,}")
print("Series found per category:")
print(final.groupby("category")["series"].nunique().reindex(CATEGORIES))
final.head()


OEM1 Typ11: matched 1,977,164 registered vehicles
OEM1 Typ12: matched 408,096 registered vehicles
OEM2 Typ21: matched 512,354 registered vehicles
OEM2 Typ22: matched 306,490 registered vehicles
Aggregated rows before labels: 442
Series found per category:
category
K1    10
K2     8
K3    10
K4     2
K5     3
K6     2
K7     2
Name: series, dtype: int64


,year,oem,vehicle_type,category,component_type,manufacturer,plant,series,n_registrations
0,2009,OEM1,Typ11,K1,K1BE1,101,1011,K1BE1-101-1011,35277
1,2009,OEM1,Typ11,K1,K1BE1,102,1021,K1BE1-102-1021,11752
2,2009,OEM1,Typ11,K1,K1BE1,104,1041,K1BE1-104-1041,70170
3,2009,OEM1,Typ11,K1,K1DI1,101,1041,K1DI1-101-1041,46102
4,2009,OEM1,Typ11,K1,K1DI1,102,1021,K1DI1-102-1021,23135


## 3. Creation of the final dataset

The app may use **only this file**. Columns include component type, manufacturer,
plant and series, with yearly registration counts.


In [35]:
# add labels and export a first version of the final CSV (all registrations)
final["label"] = final["component_type"].map(TYPE_LABEL)

# stop if an unexpected component type appears
unknown = final[final["label"].isna()]
if not unknown.empty:
    raise ValueError(f"Unlabelled component types: {unknown['component_type'].unique()}")

# placeholder clean count (= all) until the defect pipeline fills it
if "n_registrations_clean" not in final.columns:
    final["n_registrations_clean"] = final["n_registrations"]

final.to_csv(FINAL_CSV, index=False)
print(f"Wrote {FINAL_CSV}  ({len(final)} rows)")
final.head(10)


Wrote Data/SoSe26_Case_Study_finalData_Group_38.csv  (442 rows)


,year,oem,vehicle_type,category,component_type,manufacturer,plant,series,n_registrations,label
0,2009,OEM1,Typ11,K1,K1BE1,101,1011,K1BE1-101-1011,35277,Petrol engine (OEM1)
1,2009,OEM1,Typ11,K1,K1BE1,102,1021,K1BE1-102-1021,11752,Petrol engine (OEM1)
2,2009,OEM1,Typ11,K1,K1BE1,104,1041,K1BE1-104-1041,70170,Petrol engine (OEM1)
3,2009,OEM1,Typ11,K1,K1DI1,101,1041,K1DI1-101-1041,46102,Diesel engine (OEM1)
4,2009,OEM1,Typ11,K1,K1DI1,102,1021,K1DI1-102-1021,23135,Diesel engine (OEM1)
5,2009,OEM1,Typ11,K1,K1DI1,103,1031,K1DI1-103-1031,46085,Diesel engine (OEM1)
6,2009,OEM1,Typ11,K2,K2LE1,109,1091,K2LE1-109-1091,46578,Leather seats (OEM1)
7,2009,OEM1,Typ11,K2,K2ST1,109,1092,K2ST1-109-1092,185943,Fabric seats (OEM1)
8,2009,OEM1,Typ11,K3,K3AG1,105,1051,K3AG1-105-1051,13915,Automatic gearbox (OEM1)
9,2009,OEM1,Typ11,K3,K3AG1,106,1061,K3AG1-106-1061,23300,Automatic gearbox (OEM1)


### Defect-aware counts

**Defectiveness rule used here**

A registered vehicle is treated as defective when at least one of the following
is marked defective in the production data:

1. **Vehicle** — the finished car itself (`Fehlerhaft` on the Fahrzeug table).
2. **Installed component** — its engine, seats, gearbox, or body.
3. **Installed single part** — any Einzelteil that belongs to those components
   (via Bestandteile). If a part is defective, the **component that contains it
   is also treated as defective**.

Timing: a defect counts only if its defect date is **after** the vehicle’s
registration date. Earlier defects are ignored.

The final CSV then stores two parallel counts:

- `n_registrations` — all registered vehicles
- `n_registrations_clean` — same aggregation, but vehicles that fail the rule
  above are left out

Heavy reads (Komponente + Einzelteil) live in `defect_pipeline.py` /
`rebuild_final_with_defects.py` so this notebook stays readable.


In [ ]:
# build defective vehicle IDs (defectiveness rule, after registration) and rewrite final CSV
# skip recompute if the ID list already exists from a previous run
from pathlib import Path
from rebuild_final_with_defects import rebuild_final_csv
from defect_pipeline import DEFECT_IDS_PATH, build_defective_vehicle_ids

if DEFECT_IDS_PATH.exists():
    print(f"Using existing {DEFECT_IDS_PATH}")
else:
    build_defective_vehicle_ids(save=True)

final = rebuild_final_csv()
print(
    "Totals — all:",
    int(final["n_registrations"].sum()),
    " clean:",
    int(final["n_registrations_clean"].sum()),
)
final.head()


## 4. Evaluation and Result

Overall winner in a category = **series** with the highest count across all
years. Ties are reported as joint winners. Yearly charts show whether that
ranking is stable (structural demand) or changes (fashion).

The evaluation is run **twice** with the same series definition:

| Section | Count column | Meaning |
|--------|--------------|---------|
| **4.1** | `n_registrations` | Without defects — all KBA registrations |
| **4.2** | `n_registrations_clean` | With defects — exclude vehicles that are defective (vehicle, installed component, or installed part; only defects after registration) |

On trend charts there is no production start/stop flag, so:
- green triangle = series start (first registration year after 2009)
- red cross = series end (last registration year before 2016)


In [36]:
# shared helpers for both evaluation modes (without / with defects)

DATA_START_YEAR = 2009
DATA_END_YEAR = 2016
SERIES_START_MARKER = dict(
    symbol="triangle-up",
    size=8,
    color="#1F7A4D",
    line=dict(width=1, color="#1F7A4D"),
)
SERIES_END_MARKER = dict(
    symbol="x",
    size=6,
    color="#C0392B",
    line=dict(width=2, color="#C0392B"),
)


def rank_overall(count_col: str) -> pd.DataFrame:
    return (
        final.groupby(
            ["category", "component_type", "manufacturer", "plant", "series", "label"],
            as_index=False,
        )[count_col]
        .sum()
        .rename(columns={count_col: "n"})
        .sort_values(["category", "n"], ascending=[True, False])
    )


def pick_winners(overall: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for cat in CATEGORIES:
        sub = overall[overall["category"] == cat]
        top = sub["n"].max()
        rows.append(sub[sub["n"] == top])
    return pd.concat(rows, ignore_index=True)


def bar_series(overall: pd.DataFrame, category: str, title: str):
    sub = overall[overall["category"] == category].sort_values("n", ascending=False)
    fig = px.bar(
        sub,
        x="series",
        y="n",
        color="label",
        title=title,
        labels={
            "series": "Component series",
            "n": "Registered vehicles",
            "label": "Type",
        },
        color_discrete_sequence=PLOTLY_BLUES,
    )
    fig.update_layout(
        font_family="Source Sans Pro",
        plot_bgcolor="white",
        paper_bgcolor="white",
        xaxis_tickangle=-35,
    )
    return fig


def bar_bodies(overall: pd.DataFrame, title: str):
    body = overall[overall["category"].isin(["K4", "K5", "K6", "K7"])]
    fig = px.bar(
        body,
        x="series",
        y="n",
        color="category",
        title=title,
        labels={
            "series": "Body series",
            "n": "Registered vehicles",
            "category": "Category",
        },
        color_discrete_sequence=PLOTLY_BLUES,
    )
    fig.update_layout(
        font_family="Source Sans Pro",
        plot_bgcolor="white",
        paper_bgcolor="white",
        xaxis_tickangle=-35,
    )
    return fig


def mark_series_span(fig, yearly: pd.DataFrame):
    first = yearly.loc[yearly.groupby("series")["year"].idxmin()]
    started = first[first["year"] > DATA_START_YEAR]
    if not started.empty:
        fig.add_scatter(
            x=started["year"],
            y=started["n"],
            mode="markers",
            marker=SERIES_START_MARKER,
            name="series start (first registration after 2009)",
            legendgroup="series_start",
            hovertemplate="series start<br>year=%{x}<br>registrations=%{y:,}<extra></extra>",
        )

    last = yearly.loc[yearly.groupby("series")["year"].idxmax()]
    ended = last[last["year"] < DATA_END_YEAR]
    if not ended.empty:
        fig.add_scatter(
            x=ended["year"],
            y=ended["n"],
            mode="markers",
            marker=SERIES_END_MARKER,
            name="series end (last registration before 2016)",
            legendgroup="series_end",
            hovertemplate="series end<br>year=%{x}<br>registrations=%{y:,}<extra></extra>",
        )
    return fig


def trend_chart(count_col: str, category: str, title: str):
    yearly = (
        final[final["category"] == category]
        .groupby(["year", "series"], as_index=False)[count_col]
        .sum()
        .rename(columns={count_col: "n"})
    )
    fig = px.line(
        yearly,
        x="year",
        y="n",
        color="series",
        markers=True,
        title=title,
        labels={
            "year": "Registration year",
            "n": "Registered vehicles",
            "series": "Component series",
        },
        color_discrete_sequence=PLOTLY_BLUES,
    )
    fig.update_layout(
        font_family="Source Sans Pro",
        plot_bgcolor="white",
        paper_bgcolor="white",
        xaxis=dict(dtick=1),
        legend_title_text="",
    )
    return mark_series_span(fig, yearly)


def body_trend_chart(count_col: str, title: str):
    yearly = (
        final[final["category"].isin(["K4", "K5", "K6", "K7"])]
        .groupby(["year", "series", "category"], as_index=False)[count_col]
        .sum()
        .rename(columns={count_col: "n"})
    )
    fig = px.line(
        yearly,
        x="year",
        y="n",
        color="series",
        markers=True,
        title=title,
        labels={
            "year": "Registration year",
            "n": "Registered vehicles",
            "series": "Body series",
        },
        color_discrete_sequence=PLOTLY_BLUES,
    )
    fig.update_layout(
        font_family="Source Sans Pro",
        plot_bgcolor="white",
        paper_bgcolor="white",
        xaxis=dict(dtick=1),
        legend_title_text="",
    )
    return mark_series_span(fig, yearly)


def run_evaluation(count_col: str, mode_label: str):
    """Winners table + bar charts + yearly trends for one counting mode."""
    overall = rank_overall(count_col)
    winners = pick_winners(overall)

    print(f"Winners — {mode_label} (`{count_col}`):")
    display(winners)
    print(f"Total registrations in this mode: {int(overall['n'].sum()):,}")

    display(bar_series(overall, "K1", f"K1 engines — {mode_label}"))
    display(bar_series(overall, "K2", f"K2 seats — {mode_label}"))
    display(bar_series(overall, "K3", f"K3 gearboxes — {mode_label}"))
    display(bar_bodies(overall, f"Body platforms K4–K7 — {mode_label}"))

    for cat in ["K1", "K2", "K3"]:
        display(
            trend_chart(
                count_col, cat, f"Yearly registrations — {cat} — {mode_label}"
            )
        )
    display(
        body_trend_chart(
            count_col, f"Yearly registrations — body series K4–K7 — {mode_label}"
        )
    )
    return overall, winners


print("Helpers ready for sections 4.1 and 4.2.")


Overall ranking by category (top series)

category component_type manufacturer plant         series                label  n_registrations
      K1          K1BE1          104  1041 K1BE1-104-1041 Petrol engine (OEM1)           715578
      K1          K1DI1          101  1041 K1DI1-101-1041 Diesel engine (OEM1)           477052
      K1          K1DI1          103  1031 K1DI1-103-1031 Diesel engine (OEM1)           477052
      K1          K1BE1          101  1011 K1BE1-101-1011 Petrol engine (OEM1)           357789
      K1          K1BE2          104  1041 K1BE2-104-1041 Petrol engine (OEM2)           327538

category component_type manufacturer plant         series                label  n_registrations
      K2          K2ST1          109  1092 K2ST1-109-1092  Fabric seats (OEM1)           954104
      K2          K2ST1          110  1101 K2ST1-110-1101  Fabric seats (OEM1)           954104
      K2          K2ST2          109  1092 K2ST2-109-1092  Fabric seats (OEM2)           3930

,category,component_type,manufacturer,plant,series,label,n_registrations
0,K1,K1BE1,104,1041,K1BE1-104-1041,Petrol engine (OEM1),715578
1,K2,K2ST1,109,1092,K2ST1-109-1092,Fabric seats (OEM1),954104
2,K2,K2ST1,110,1101,K2ST1-110-1101,Fabric seats (OEM1),954104
3,K3,K3SG1,107,1071,K3SG1-107-1071,Manual gearbox (OEM1),1144925
4,K4,K4,114,1141,K4-114-1141,Body (OEM1 Typ11),1186299
5,K5,K5,112,1122,K5-112-1122,Body (OEM1 Typ12),244857
6,K6,K6,113,1132,K6-113-1132,Body (OEM2 Typ21),256177
7,K6,K6,114,1142,K6-114-1142,Body (OEM2 Typ21),256177
8,K7,K7,113,1132,K7-113-1132,Body (OEM2 Typ22),153245
9,K7,K7,114,1142,K7-114-1142,Body (OEM2 Typ22),153245


### 4.1 Without defects

All KBA registrations (`n_registrations`). This is the baseline popularity ranking.


In [37]:
# 4.1 — evaluation and graphs WITHOUT defects (all registrations)
overall_all, winners_all = run_evaluation(
    "n_registrations",
    "without defects (all registrations)",
)


### 4.2 With defects

Same charts and winner logic, but using `n_registrations_clean`: vehicles that
count as defective under the rule above are excluded (defects after registration
only). This can break ties and change the recommended series.


In [41]:
# 4.2 — evaluation and graphs WITH defective vehicles excluded
overall_clean, winners_clean = run_evaluation(
    "n_registrations_clean",
    "with defective vehicles excluded",
)

# where the winner set differs between the two modes
cmp = winners_all.merge(
    winners_clean,
    on=["category", "series"],
    how="outer",
    suffixes=("_all", "_clean"),
    indicator=True,
)
changed = cmp[cmp["_merge"] != "both"].copy()
if changed.empty:
    print("Same winner set in both modes.")
else:
    print("Winner set differs between modes (left_only = without defects only, "
          "right_only = with defects only):")
    display(
        changed[
            [
                "category",
                "series",
                "n_all",
                "n_clean",
                "_merge",
            ]
        ].sort_values(["category", "series"])
    )


### Recommendation

#### Without defects (`n_registrations`)

- **K1:** `K1BE1-104-1041` (petrol engine, manufacturer 104, plant 1041)
- **K2:** `K2ST1-109-1092` and `K2ST1-110-1101` (tie — fabric seats, two plants)
- **K3:** `K3SG1-107-1071` (manual gearbox, manufacturer 107, plant 1071)
- **K4:** `K4-114-1141`
- **K5:** `K5-112-1122`
- **K6:** `K6-113-1132` and `K6-114-1142` (tie)
- **K7:** `K7-113-1132` and `K7-114-1142` (tie)

Among bodies, **K4-114-1141** has the most registrations overall.

**One car to build (feasible mix):** body `K4-114-1141`, seats from the K2 tie
(`K2ST1-109-1092` or `K2ST1-110-1101`), gearbox `K3SG1-107-1071`, engine
`K1BE1-104-1041`. Do not fit K4–K7 together.

#### With defects excluded (`n_registrations_clean`)

Excluding defective vehicles (rule above) breaks several ties:

- **K1:** `K1BE1-104-1041` (unchanged)
- **K2:** `K2ST1-110-1101` only (tie broken vs `K2ST1-109-1092`)
- **K3:** `K3SG1-107-1071` (unchanged)
- **K4:** `K4-114-1141` (unchanged)
- **K5:** `K5-112-1122` (unchanged)
- **K6:** `K6-114-1142` only (tie broken vs `K6-113-1132`)
- **K7:** `K7-113-1132` only (tie broken vs `K7-114-1142`)

**One car to build under the defect-aware mode:** body `K4-114-1141`, seats
`K2ST1-110-1101`, gearbox `K3SG1-107-1071`, engine `K1BE1-104-1041`.

### Web app

The Streamlit app reads **only** `Data/SoSe26_Case_Study_finalData_Group_38.csv`
and switches between the same two modes with a toggle.

```text
python3 -m streamlit run SoSe26_Case_Study_App_Group_38.py
```

Design: light blue, Source Sans Pro (`www/fonts/`), logo in `www/img/logo.svg`.
Add screenshots of the app tabs to `Additional_files/` before submission.
